In [2]:
import pandas as pd
import geopandas as gpd

In [10]:
df = pd.read_csv("NER_landslide_cleaned_full.csv")

In [11]:
gdf_landslides = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",
)

In [7]:
soil_map = gpd.read_file("INDIA_SOIL_MAP_FAO.geojson")

In [13]:
if soil_map.crs != gdf_landslides.crs:
    soil_map = soil_map.to_crs(gdf_landslides.crs)

In [14]:
df_with_soil = gpd.sjoin(
    gdf_landslides, soil_map, how="left", predicate="intersects"
)

In [15]:
df_with_soil = df_with_soil.drop(columns=["geometry"])
df_with_soil.to_csv("NER_landslide_with_soil.csv", index=False)
print("Spatial join complete. Saved to NER_landslide_with_soil.csv")
print(f"Columns added: {[c for c in soil_map.columns if c != 'geometry']}")

Spatial join complete. Saved to NER_landslide_with_soil.csv
Columns added: ['SNUM', 'FAOSOIL', 'DOMSOI', 'PHASE1', 'PHASE2', 'MISCLU1', 'MISCLU2', 'PERMAFROST', 'CNTCODE', 'CNTNAME', 'SQKM', 'COUNTRY']


In [16]:
garbage_cols = [
    'index_right', 'SNUM', 'COUNTRY', 'CNTCODE', 'CNTNAME', 
    'SQKM', 'PERMAFROST', 'PHASE1', 'PHASE2', 'MISCLU1', 'MISCLU2'
]

In [17]:
df_clean = df.drop(columns=garbage_cols, errors='ignore')

In [18]:
df_clean.to_csv("NER_landslide_final_ML.csv", index=False)

In [19]:
print("Cleaned Dataset Shape:", df_clean.shape)
print("Soil data secured in 'DOMSOI' and 'FAOSOIL'.")

Cleaned Dataset Shape: (351, 31)
Soil data secured in 'DOMSOI' and 'FAOSOIL'.


In [20]:
df = pd.read_csv("NER_landslide_with_soil.csv")

In [21]:
garbage_cols = [
    'index_right', 'SNUM', 'COUNTRY', 'CNTCODE', 'CNTNAME', 
    'SQKM', 'PERMAFROST', 'PHASE1', 'PHASE2', 'MISCLU1', 'MISCLU2'
]

In [22]:
df_clean = df.drop(columns=garbage_cols, errors='ignore')

In [23]:
df_clean.to_csv("NER_landslide_final_ML.csv", index=False)

print("Cleaned Dataset Shape:", df_clean.shape)
print("Soil data secured in 'DOMSOI' and 'FAOSOIL'.")

Cleaned Dataset Shape: (351, 33)
Soil data secured in 'DOMSOI' and 'FAOSOIL'.


In [4]:
#now creating the data of soil for the road map

In [15]:
"""
assign_corridor_soil.py
---------------------------
The corridor edges (guwahati_shillong_corridor_final.csv) don't have soil
data - only the original 351 landslide points do (DOMSOI column, raw text).

Instead of a new external soil API (which would use a different
classification system and NOT match the model's trained categories),
this assigns each corridor point the DOMSOI value of its NEAREST known
soil-labeled point from the landslide dataset - geologically reasonable
(nearby locations tend to share soil type), and guarantees category
consistency with the trained model.

Then applies the SAME one-hot encoding as training, and makes sure every
column the model expects exists (filling any category the corridor
subset doesn't happen to have with 0s) - critical for matching
model_feature_columns.pkl exactly.

Run this AFTER calculate_corridor_slope.py and AFTER train_risk_model.py
(needs model_feature_columns.pkl to know the exact expected columns).
"""

import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import joblib

CORRIDOR_INPUT_CSV = "guwahati_shillong_edges.csv"
SOIL_REFERENCE_CSV = "NER_landslide_final_ML.csv"   # has raw latitude/longitude/DOMSOI
FEATURE_COLUMNS_PATH = "model_feature_columns.pkl"
OUTPUT_CSV = "guwahati_shillong_edges.csv"

MIN_COUNT = 20  # must match the value used in train_risk_model.py


def assign_nearest_soil(corridor_df, reference_df):
    """
    For each corridor edge (using its midpoint), finds the nearest
    landslide-dataset point with a known DOMSOI value and copies it over.
    """
    reference_df = reference_df.dropna(subset=["DOMSOI"]).reset_index(drop=True)

    # Corridor edge midpoint = average of start/end coordinates
    corridor_df["mid_lat"] = (corridor_df["start_lat"] + corridor_df["end_lat"]) / 2
    corridor_df["mid_lon"] = (corridor_df["start_lon"] + corridor_df["end_lon"]) / 2

    # Build a KD-tree of all known soil points for fast nearest-neighbor lookup
    reference_points = reference_df[["latitude", "longitude"]].values
    tree = cKDTree(reference_points)

    corridor_points = corridor_df[["mid_lat", "mid_lon"]].values
    distances, nearest_idx = tree.query(corridor_points)

    corridor_df["DOMSOI"] = reference_df.loc[nearest_idx, "DOMSOI"].values
    corridor_df["soil_match_distance_deg"] = distances  # rough sanity-check column

    return corridor_df


def encode_soil_matching_training(df, expected_columns, soil_col="DOMSOI", min_count=MIN_COUNT):
    """
    One-hot encodes DOMSOI, then aligns columns EXACTLY to what the
    trained model expects - adding any missing DOMSOI_* category as
    all-zero, and dropping anything extra.
    """
    df = df.copy()

    value_counts = df[soil_col].value_counts()
    rare_categories = value_counts[value_counts < min_count].index.tolist()
    df[f"{soil_col}_grouped"] = df[soil_col].apply(
        lambda x: "Other" if x in rare_categories else x
    )

    dummies = pd.get_dummies(df[f"{soil_col}_grouped"], prefix=soil_col, dtype=int)
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(columns=[f"{soil_col}_grouped", soil_col])

    # Ensure every DOMSOI_* column the model expects exists, even if this
    # small corridor subset didn't happen to contain that soil category
    expected_soil_cols = [c for c in expected_columns if c.startswith(f"{soil_col}_")]
    for col in expected_soil_cols:
        if col not in df.columns:
            df[col] = 0
            print(f"  Added missing category column '{col}' (not present in corridor, filled with 0)")

    return df


def main():
    corridor_df = pd.read_csv(CORRIDOR_INPUT_CSV)
    reference_df = pd.read_csv(SOIL_REFERENCE_CSV)
    expected_columns = joblib.load(FEATURE_COLUMNS_PATH)

    print(f"Corridor edges: {len(corridor_df)}")
    print(f"Reference soil points: {len(reference_df)}")

    corridor_df = assign_nearest_soil(corridor_df, reference_df)

    print("\nDOMSOI assigned to corridor (before encoding):")
    print(corridor_df["DOMSOI"].value_counts())

    print("\nSanity check - distance to nearest match (degrees, ~111km per degree):")
    print(corridor_df["soil_match_distance_deg"].describe())

    corridor_df = encode_soil_matching_training(corridor_df, expected_columns)

    corridor_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved to {OUTPUT_CSV}")
    print(f"Final columns: {list(corridor_df.columns)}")


if __name__ == "__main__":
    main()

Corridor edges: 175
Reference soil points: 351

DOMSOI assigned to corridor (before encoding):
DOMSOI
Ao    137
Nd     38
Name: count, dtype: int64

Sanity check - distance to nearest match (degrees, ~111km per degree):
count    175.000000
mean       0.041772
std        0.042525
min        0.001343
25%        0.008280
50%        0.031069
75%        0.059093
max        0.190851
Name: soil_match_distance_deg, dtype: float64
  Added missing category column 'DOMSOI_Af' (not present in corridor, filled with 0)
  Added missing category column 'DOMSOI_Bd' (not present in corridor, filled with 0)
  Added missing category column 'DOMSOI_Be' (not present in corridor, filled with 0)
  Added missing category column 'DOMSOI_Bh' (not present in corridor, filled with 0)
  Added missing category column 'DOMSOI_Gd' (not present in corridor, filled with 0)
  Added missing category column 'DOMSOI_Ge' (not present in corridor, filled with 0)
  Added missing category column 'DOMSOI_I' (not present in corri